In [1]:
import os
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
import pandas as pd
import datetime
from torch.utils.tensorboard import SummaryWriter

if torch.cuda.is_available():
    print("GPU is available")
    print(f"Using device: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available, using CPU")
    
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# データフォルダ
standardized_data_folder = "/home/nishioka/GNN/Defect_4x4_Normalized1"
label_data_folder = "/home/nishioka/GNN/DefectClass_OneHot_test1"

# 座標データのロード
x_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/x_2layer_normalized.npy")[:3654]
y_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/y_2layer_normalized.npy")[:3654]
z_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/z_2layer_normalized.npy")[:3654]

# エッジ情報を読み込む
edges = np.load("/home/nishioka/GNN/BasicdataforGNN/edges_2layer.npy")
edge_index = torch.tensor(edges.T, dtype=torch.long).to(device)

# データとラベルファイルを対応付け
data_files = [f for f in os.listdir(standardized_data_folder) if f.startswith("Normalized1_Defect4x4_ELNOD")]
label_files = [f for f in os.listdir(label_data_folder) if f.startswith("DefectClass_L")]

train_pairs = []
val_pairs = []
test_pairs = []

# 欠陥なしデータのペア
defect_free_pair = ("/home/nishioka/GNN/BasicdataforGNN/Normalized1_nodefect_ElNOD.npy",
                    "/home/nishioka/GNN/BasicdataforGNN/DefectClass_nodefect.npy")

# 欠陥なしデータを追加
for _ in range(80):
    train_pairs.append(defect_free_pair)

for _ in range(10):
    val_pairs.append(defect_free_pair)
    test_pairs.append(defect_free_pair)

# データとラベルのペアを取得する関数
def extract_layer_block(file_name):
    if file_name.startswith("0"):
        return (0, 0)  # 欠陥なしデータの場合
    try:
        layer_block_str = file_name.split("_")[-1].replace(".npy", "")
        layer_str = layer_block_str.split("L")[1].split("B")[0]
        block_str = layer_block_str.split("B")[1]
        layer = int(layer_str)
        block = int(block_str)
        return (layer, block)
    except (ValueError, IndexError):
        print(f"無効なファイル名の形式: {file_name}")
        return None

# データとラベルのペアを作成
data_label_pairs = {}
for data_file in data_files:
    layer_block = extract_layer_block(data_file)
    if layer_block:
        data_label_pairs[layer_block] = {"data": data_file}

for label_file in label_files:
    layer_block = extract_layer_block(label_file)
    if layer_block and layer_block in data_label_pairs:
        data_label_pairs[layer_block]["label"] = label_file

# データとラベルのペアを作成する関数
def prepare_data(pairs):
    sampled_data = []
    sampled_labels = []
    
    for data_file, label_file in pairs:
        data_file_path = os.path.join(standardized_data_folder, data_file)
        label_file_path = os.path.join(label_data_folder, label_file)

        # データとラベルを読み込む
        values = np.load(data_file_path)[:3654]
        label = np.load(label_file_path)[:3654]
        
        # 座標データと応力データを結合してノード特徴量を作成
        node_features = np.vstack((x_coords, y_coords, z_coords, values)).T
        sampled_data.append(node_features)
        sampled_labels.append(label)
    
    # データを結合しテンソルに変換
    sampled_data = np.stack(sampled_data)
    sampled_labels = np.hstack(sampled_labels)

    x = torch.tensor(sampled_data.reshape(-1, 4), dtype=torch.float).to(device)
    y = torch.tensor(sampled_labels, dtype=torch.long).to(device)  # 分類のため long 型に変更

    return x, y

# 有効なペアを取得し、欠陥なしデータを追加
valid_pairs = [(v["data"], v["label"]) for k, v in data_label_pairs.items() if "label" in v]


all_x, all_y = prepare_data(valid_pairs)

# 全体データからの DataLoader 用のサンプルリスト作成
all_data_list = []
num_all_samples = all_x.shape[0] // 3654
for i in range(num_all_samples):
    all_data_list.append(Data(x=all_x[i * 3654:(i + 1) * 3654], edge_index=edge_index, y=all_y[i * 3654:(i + 1) * 3654]))

# データとラベルのペアを作成
pairs = list(zip(data_files, label_files))

# 交差検証の設定
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_fold_metrics = []
for train_index, val_index in kf.split(pairs):
    print(f"Starting Fold {fold}")
    train_pairs = [pairs[i] for i in train_index]
    val_pairs = [pairs[i] for i in val_index]

    # データ準備
    train_x, train_y = prepare_data(train_pairs)
    val_x, val_y = prepare_data(val_pairs)

    # Dataオブジェクトにまとめる
    train_data = Data(x=train_x, edge_index=edge_index, y=train_y)
    val_data = Data(x=val_x, edge_index=edge_index, y=val_y)

    # GCNモデルの定義（分類用）
    class GCNModel(torch.nn.Module):
        def __init__(self, hidden_channels=64, num_classes=18):
            super(GCNModel, self).__init__()
            self.conv1 = GCNConv(4, hidden_channels)
            self.conv2 = GCNConv(hidden_channels, hidden_channels * 2)
            self.conv3 = GCNConv(hidden_channels * 2, hidden_channels * 4)
            self.conv4 = GCNConv(hidden_channels * 4, hidden_channels * 2)
            self.conv5 = GCNConv(hidden_channels * 2, hidden_channels)
            self.fc = nn.Linear(hidden_channels, num_classes)
            self.dropout = nn.Dropout(p=0.2)  # ドロップアウト率

        def forward(self, data):
            x, edge_index = data.x, data.edge_index
            x = F.relu(self.conv1(x, edge_index))
            x = self.dropout(x)
            x = F.relu(self.conv2(x, edge_index))
            x = self.dropout(x)
            x = F.relu(self.conv3(x, edge_index))
            x = self.dropout(x)
            x = F.relu(self.conv4(x, edge_index))
            x = self.dropout(x)
            x = F.relu(self.conv5(x, edge_index))
            x = self.fc(x)
            return F.log_softmax(x, dim=1)

    # ハイパーパラメータ設定
    hidden_channels = 32
    learning_rate = 0.0005
    batch_size = 2
    epochs = 50
    weight_decay = 5e-4
    patience = 50  # Early Stopping用

    # モデル、損失関数、オプティマイザの定義
    model = GCNModel(hidden_channels=hidden_channels, num_classes=18)
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for training")
        model = nn.DataParallel(model)
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()  # 分類用にクロスエントロピー損失を使用

    # データローダーの作成
    train_loader = DataLoader([train_data], batch_size=batch_size, shuffle=True)
    val_loader = DataLoader([val_data], batch_size=batch_size, shuffle=False)

    # 学習ループの開始
    best_val_loss = float('inf')
    counter = 0
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            loss = loss_fn(out, batch.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        train_loss = total_loss / len(train_loader)

        # 検証ステップ
        model.eval()
        val_loss = 0
        correct = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch)
                loss = loss_fn(out, batch.y)
                val_loss += loss.item()
                pred = out.argmax(dim=1)
                correct += (pred == batch.y).sum().item()

        val_loss /= len(val_loader)
        val_accuracy = correct / batch.y.size(0)

        # 結果の表示
        print(f'Fold {fold}, Epoch {epoch}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')

        # Early Stopping の条件確認
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodel/gcn_model_best_fold{fold}_{timestamp}.pth')
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping triggered for Fold {fold}")
                break

    # Foldごとの結果を保存
    all_fold_metrics.append({
        "fold": fold,
        "best_val_loss": best_val_loss,
        "val_accuracy": val_accuracy
    })

    print(f"Training finished for Fold {fold}.")
    fold += 1

# 最終テスト用データを準備
train_val_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)
train_val_x, train_val_y = prepare_data(train_val_pairs)
test_x, test_y = prepare_data(test_pairs)

# Dataオブジェクトにまとめる
train_val_data = Data(x=train_val_x, edge_index=edge_index, y=train_val_y)
test_data = Data(x=test_x, edge_index=edge_index, y=test_y)

# モデル再定義
final_model = GCNModel(hidden_channels=hidden_channels).to(device)

# final_model = GCNModel(hidden_channels=hidden_channels, num_classes=18)
# if torch.cuda.device_count() > 1:
#     print(f"Using {torch.cuda.device_count()} GPUs for training")
#     final_model = nn.DataParallel(model)
# final_model = model.to(device)

final_optimizer = torch.optim.Adam(final_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
final_loss_fn = nn.CrossEntropyLoss()

# トレーニングと検証データで最終モデルをトレーニング
final_train_loader = DataLoader([train_val_data], batch_size=batch_size, shuffle=True)
final_test_loader = DataLoader([test_data], batch_size=batch_size, shuffle=False)

best_val_loss = float('inf')
final_model.train()
for epoch in range(1, epochs + 1):
    total_loss = 0
    for batch in final_train_loader:
        batch = batch.to(device)
        final_optimizer.zero_grad()
        out = final_model(batch)
        loss = final_loss_fn(out, batch.y)
        loss.backward()
        final_optimizer.step()
        total_loss += loss.item()
    
    train_loss = total_loss / len(final_train_loader)
    print(f"Epoch {epoch}, Train Loss: {train_loss:.4f}")

# テストデータで最終モデルの評価
final_model.eval()
correct = 0
total_loss = 0
with torch.no_grad():
    for batch in final_test_loader:
        batch = batch.to(device)
        out = final_model(batch)
        loss = final_loss_fn(out, batch.y)
        total_loss += loss.item()
        pred = out.argmax(dim=1)
        correct += (pred == batch.y).sum().item()

test_loss = total_loss / len(final_test_loader)
test_accuracy = correct / batch.y.size(0)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

GPU is available
Using device: NVIDIA GeForce RTX 3090
Starting Fold 1
Using 2 GPUs for training


OutOfMemoryError: Caught OutOfMemoryError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/home/nishioka/miniconda3/envs/interstage_gnn_first/lib/python3.12/site-packages/torch/nn/parallel/parallel_apply.py", line 83, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nishioka/miniconda3/envs/interstage_gnn_first/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1511, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nishioka/miniconda3/envs/interstage_gnn_first/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1520, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1244760/3811516021.py", line 163, in forward
    x = F.relu(self.conv2(x, edge_index))
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nishioka/miniconda3/envs/interstage_

In [20]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split, KFold
import datetime

# Device setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Data folders
standardized_data_folder = "/home/nishioka/GNN/Defect_4x4_Normalized1"
label_data_folder = "/home/nishioka/GNN/DefectClass_OneHot_test1"

# Load coordinate data
x_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/x_2layer_normalized.npy")[:3654]
y_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/y_2layer_normalized.npy")[:3654]
z_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/z_2layer_normalized.npy")[:3654]

# Load edge information
edges = np.load("/home/nishioka/GNN/BasicdataforGNN/edges_2layer.npy")
edge_index = torch.tensor(edges.T, dtype=torch.long)

def prepare_data(pairs):
    data_list = []
    for data_file, label_file in pairs:
        data_file_path = os.path.join(standardized_data_folder, data_file)
        label_file_path = os.path.join(label_data_folder, label_file)

        # Load data and labels
        values = np.load(data_file_path)[:3654]
        label = np.load(label_file_path)[:3654]

        # Create node features
        node_features = np.vstack((x_coords, y_coords, z_coords, values)).T
        x = torch.tensor(node_features, dtype=torch.float)
        y = torch.tensor(label, dtype=torch.long).squeeze()

        data = Data(x=x, edge_index=edge_index, y=y)
        data_list.append(data)
    return data_list

# Prepare dataset
data_files = [f for f in os.listdir(standardized_data_folder) if f.startswith("Normalized1_Defect4x4_ELNOD")]
label_files = [f for f in os.listdir(label_data_folder) if f.startswith("DefectClass_L")]
pairs = list(zip(data_files, label_files))

# Model definition
class GCNModel(torch.nn.Module):
    def __init__(self, hidden_channels=32, num_classes=18):
        super(GCNModel, self).__init__()
        self.conv1 = GCNConv(4, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels * 2)
        self.conv3 = GCNConv(hidden_channels * 2, hidden_channels)
        self.fc = nn.Linear(hidden_channels, num_classes)
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv3(x, edge_index))
        x = self.fc(x)
        return F.log_softmax(x, dim=1)

# Hyperparameters
hidden_channels = 32
learning_rate = 0.0005
batch_size = 2  # Adjust as needed
epochs = 10
weight_decay = 5e-4
patience = 50  # Early stopping

# Loss function
loss_fn = nn.CrossEntropyLoss()

# Initialize model
model = GCNModel(hidden_channels=hidden_channels, num_classes=18).to(device)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for training")
    model = nn.DataParallel(model, device_ids=[0, 1])

# Cross-validation setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_fold_metrics = []
for train_index, val_index in kf.split(pairs):
    print(f"Starting Fold {fold}")
    train_pairs = [pairs[i] for i in train_index]
    val_pairs = [pairs[i] for i in val_index]

    # Prepare data
    train_dataset = prepare_data(train_pairs)
    val_dataset = prepare_data(val_pairs)

    # Data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Training loop
    best_val_loss = float('inf')
    counter = 0
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0

        for batch in train_loader:
            batch = batch.to(device)  # Move batch to device
            optimizer.zero_grad()
            out = model(batch)
            y = batch.y
            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)

        # Validation
        model.eval()
        val_loss = 0
        correct = 0
        total_samples = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch)
                y = batch.y
                loss = loss_fn(out, y)
                val_loss += loss.item()
                pred = out.argmax(dim=1)
                correct += (pred == y).sum().item()
                total_samples += y.size(0)

        val_loss /= len(val_loader)
        val_accuracy = correct / total_samples

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping triggered for Fold {fold}")
                break

        print(f'Fold {fold}, Epoch {epoch}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}')

    # Save metrics
    all_fold_metrics.append({
        "fold": fold,
        "best_val_loss": best_val_loss,
        "val_accuracy": val_accuracy
    })

    fold += 1

# Final training on all data
train_val_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)
train_val_dataset = prepare_data(train_val_pairs)
test_dataset = prepare_data(test_pairs)

final_train_loader = DataLoader(train_val_dataset, batch_size=batch_size, shuffle=True)
final_test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Starting final training on all data...")
model.train()
for epoch in range(1, epochs + 1):
    total_loss = 0
    for batch in final_train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        y = batch.y
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(final_train_loader)
    print(f"Epoch {epoch}, Train Loss: {train_loss:.4f}")

print("Evaluating on test data...")
model.eval()
correct = 0
total_loss = 0
total_samples = 0
with torch.no_grad():
    for batch in final_test_loader:
        batch = batch.to(device)
        out = model(batch)
        y = batch.y
        loss = loss_fn(out, y)
        total_loss += loss.item()
        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()
        total_samples += y.size(0)

test_loss = total_loss / len(final_test_loader)
test_accuracy = correct / total_samples
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


Using 2 GPUs for training
Starting Fold 1


RuntimeError: Caught RuntimeError in replica 1 on device 1.
Original Traceback (most recent call last):
  File "/home/nishioka/miniconda3/envs/interstage_gnn_first/lib/python3.12/site-packages/torch/nn/parallel/parallel_apply.py", line 83, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nishioka/miniconda3/envs/interstage_gnn_first/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1511, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nishioka/miniconda3/envs/interstage_gnn_first/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1520, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_2854430/2203474051.py", line 64, in forward
    x = F.relu(self.conv1(x, edge_index))
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nishioka/miniconda3/envs/interstage_gnn_first

In [17]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split, KFold
import matplotlib.pyplot as plt
import pandas as pd
import datetime

# Device setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Data folders
standardized_data_folder = "/home/nishioka/GNN/Defect_4x4_Normalized1"
label_data_folder = "/home/nishioka/GNN/DefectClass_OneHot_test1"

# Load coordinate data
x_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/x_2layer_normalized.npy")[:3654]
y_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/y_2layer_normalized.npy")[:3654]
z_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/z_2layer_normalized.npy")[:3654]

# Load edge information
edges = np.load("/home/nishioka/GNN/BasicdataforGNN/edges_2layer.npy")
edge_index = torch.tensor(edges.T, dtype=torch.long)

def prepare_data(pairs):
    data_list = []
    for idx, (data_file, label_file) in enumerate(pairs):
        # Print progress every 100 samples
        if idx % 100 == 0:
            print(f"Preparing data sample {idx+1}/{len(pairs)}")
        data_file_path = os.path.join(standardized_data_folder, data_file)
        label_file_path = os.path.join(label_data_folder, label_file)

        # Load data and labels
        values = np.load(data_file_path)[:3654]
        label = np.load(label_file_path)[:3654]

        # Convert one-hot encoded labels to class indices
        label = label.argmax(axis=1)

        # Create node features
        node_features = np.vstack((x_coords, y_coords, z_coords, values)).T
        x = torch.tensor(node_features, dtype=torch.float)
        y = torch.tensor(label, dtype=torch.long).squeeze()

        data = Data(x=x, edge_index=edge_index, y=y)
        data_list.append(data)
    return data_list

# Prepare dataset
data_files = [f for f in os.listdir(standardized_data_folder) if f.startswith("Normalized1_Defect4x4_ELNOD")]
label_files = [f for f in os.listdir(label_data_folder) if f.startswith("DefectClass_L")]
pairs = list(zip(data_files, label_files))
print(f"Total data-label pairs: {len(pairs)}")

# Model definition
class GCNModel(torch.nn.Module):
    def __init__(self, hidden_channels=32, num_classes=18):
        super(GCNModel, self).__init__()
        self.conv1 = GCNConv(4, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels * 2)
        self.conv3 = GCNConv(hidden_channels * 2, hidden_channels)
        self.fc = nn.Linear(hidden_channels, num_classes)
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = x.to(device)
        edge_index = edge_index.to(device)
        # Forward pass
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv3(x, edge_index))
        x = self.fc(x)
        return F.log_softmax(x, dim=1)

# Hyperparameters
hidden_channels = 32
learning_rate = 0.005
batch_size = 2  # Adjusted to 1 to reduce memory usage
epochs = 10
weight_decay = 5e-4
patience = 50  # Early stopping

# Loss function
loss_fn = nn.NLLLoss()  # Using NLLLoss because output is log_softmax

# Initialize model on the default device
model = GCNModel(hidden_channels=hidden_channels, num_classes=18).to(device)

# Cross-validation setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_fold_metrics = []
for train_index, val_index in kf.split(pairs):
    print(f"Starting Fold {fold}")
    train_pairs = [pairs[i] for i in train_index]
    val_pairs = [pairs[i] for i in val_index]

    # Prepare data
    train_dataset = prepare_data(train_pairs)
    val_dataset = prepare_data(val_pairs)

    # Data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Training loop
    best_val_loss = float('inf')
    counter = 0
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            y = batch.y.to(device)
            # Reshape y to be 1D
            y = y.view(-1)
            # Compute loss
            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)

        # Validation
        model.eval()
        val_loss = 0
        correct = 0
        total_samples = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch)
                y = batch.y.to(device)
                y = y.view(-1)
                loss = loss_fn(out, y)
                val_loss += loss.item()
                pred = out.argmax(dim=1)
                correct += (pred == y).sum().item()
                total_samples += y.size(0)

        val_loss /= len(val_loader)
        val_accuracy = correct / total_samples if total_samples > 0 else 0

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            counter = 0
            # Save the best model
            torch.save(model.state_dict(), f"best_model_fold_{fold}.pth")
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping triggered for Fold {fold}")
                break

        print(f'Fold {fold}, Epoch {epoch}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}')

    # Save metrics
    all_fold_metrics.append({
        "fold": fold,
        "best_val_loss": best_val_loss,
        "val_accuracy": val_accuracy
    })

    fold += 1

# Final training on all data
train_val_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)
train_val_dataset = prepare_data(train_val_pairs)
test_dataset = prepare_data(test_pairs)

final_train_loader = DataLoader(train_val_dataset, batch_size=batch_size, shuffle=True)
final_test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Starting final training on all data...")
model.train()
for epoch in range(1, epochs + 1):
    total_loss = 0
    for batch in final_train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        y = batch.y.to(device)
        y = y.view(-1)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(final_train_loader)
    print(f"Epoch {epoch}, Train Loss: {train_loss:.4f}")

print("Evaluating on test data...")
model.eval()
correct = 0
total_loss = 0
total_samples = 0
with torch.no_grad():
    for batch in final_test_loader:
        batch = batch.to(device)
        out = model(batch)
        y = batch.y.to(device)
        y = y.view(-1)
        loss = loss_fn(out, y)
        total_loss += loss.item()
        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()
        total_samples += y.size(0)

test_loss = total_loss / len(final_test_loader)
test_accuracy = correct / total_samples if total_samples > 0 else 0
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

# Optionally, print fold metrics
print("Fold Metrics:")
for metrics in all_fold_metrics:
    print(metrics)


Using device: cuda:0
Total data-label pairs: 1619
Starting Fold 1
Preparing data sample 1/1295
Preparing data sample 101/1295
Preparing data sample 201/1295
Preparing data sample 301/1295
Preparing data sample 401/1295
Preparing data sample 501/1295
Preparing data sample 601/1295
Preparing data sample 701/1295
Preparing data sample 801/1295
Preparing data sample 901/1295
Preparing data sample 1001/1295
Preparing data sample 1101/1295
Preparing data sample 1201/1295
Preparing data sample 1/324
Preparing data sample 101/324
Preparing data sample 201/324
Preparing data sample 301/324
Fold 1, Epoch 1, Train Loss: 0.2602, Val Loss: 0.1064, Val Acc: 0.9871
Fold 1, Epoch 2, Train Loss: 0.1077, Val Loss: 0.1056, Val Acc: 0.9871
Fold 1, Epoch 3, Train Loss: 0.1071, Val Loss: 0.1052, Val Acc: 0.9871
Fold 1, Epoch 4, Train Loss: 0.1066, Val Loss: 0.1057, Val Acc: 0.9871
Fold 1, Epoch 5, Train Loss: 0.1063, Val Loss: 0.1053, Val Acc: 0.9871
Fold 1, Epoch 6, Train Loss: 0.1062, Val Loss: 0.1050, Va

In [19]:
import torch 
torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodel/10classmodel/gcn_model_final_{timestamp}.pth')
